In [43]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils.text import extract_text
import re
import json
import pandas as pd

In [44]:
ROOT = Path("~/code/selector_examenes").expanduser()
FORMULATION_PATH = Path("Formulación")

In [45]:
# file = "Formulación/2020 - Formulación.pdf"

In [46]:
def extraer_nombres_formulas(statement):
    patron = re.compile(r"R\s*E\s*S\s*O\s*L\s*U\s*C\s*I\s*[ÓO]\s*N", re.IGNORECASE)
    m = patron.search(statement)
    end_position = m.start() if m else -1
    compounds_end_position = statement.lower().find("d)")

    # extraer nombre examen
    statement = statement[:end_position]
    lines = statement.strip().split("\n")
    exam_name = lines[-1].lower()
    exam_name = re.sub(r"ejercicio \w+", "", exam_name)

    # compuestos a nombrar
    compounds = statement[:compounds_end_position]
    compounds = compounds.replace("Formule o nombre los compuestos siguientes: ", "").strip()
    return exam_name, compounds

In [47]:
formulation_statements = {}
for file in FORMULATION_PATH.iterdir():
    if file.is_file():
        with pdfplumber.open(file) as pdf:
            year = int(file.name[:4])
            formulation_statements[year] = []
            for page in pdf.pages[1:]:
                statement = page.extract_text()
                formulation_statements[year].append(extraer_nombres_formulas(statement))

In [48]:
formulation_statements

{2016: [('química. 2016. junio. . opción a',
   'a) Hidróxido de niquel(III); b) Ácido peryódico;\nc) Nitrobenceno;'),
  ('química. 2016. junio. . opción b',
   'a) Nitruro de aluminio; b) Hidrogenocromato de\ncobre(II); c) 3-metilbut-1-ino;'),
  ('química. 2016. reserva 1. . opción a',
   'Formule o nombre los siguientes compuestos: a) Óxido de vanadio(V); b) Ácido nitroso;\nc) Metilpropeno;'),
  ('química. 2016. reserva 1. . opción b',
   'Formule o nombre los siguientes compuestos: a) Hidróxido de berilio; b) Permanganato de\nbario; c) Propanoato de metilo;'),
  ('química. 2016. reserva 2. . opción a',
   'Formule o nombre los siguientes compuestos: a) Carbonato de rubidio; b) Sulfuro de cobre(II);\nc) Penta-1,3-dieno;'),
  ('química. 2016. reserva 2. . opción b',
   'Formule o nombre los siguientes compuestos: a) Óxido de plomo(IV); b) Ácido sulfuroso;\nc) Etanoato de propilo;'),
  ('química. 2016. reserva 3. . opción a',
   'Formule o nombre los siguientes compuestos: a) Fluoruro 

In [78]:
formulation_statements[2023]

[('', ''),
 ('', ''),
 ('', ''),
 ('', ''),
 ('', ''),
 ('', ''),
 ('', ''),
 ('', ''),
 ('', ''),
 ('', ''),
 ('', ''),
 ('', '')]

In [74]:
form_emestrada = []
for year, exercises in formulation_statements.items():
    for ex in exercises:
        form_emestrada.append({"year" : year, "exam" : ex[0], "statement" : ex[1]})

In [75]:
form_emestrada = pd.DataFrame(form_emestrada)

In [77]:
form_emestrada.iloc[indices]

,year,exam,statement
70,2025,,
71,2025,,
72,2025,,
73,2025,,
74,2023,,
75,2023,,
76,2023,,
77,2023,,
78,2023,,
79,2023,,


Limpiar texto de `exam`

In [51]:
form_emestrada.exam.head()

0        química. 2016. junio. . opción a
1        química. 2016. junio. . opción b
2    química. 2016. reserva 1. . opción a
3    química. 2016. reserva 1. . opción b
4    química. 2016. reserva 2. . opción a
Name: exam, dtype: str

In [52]:
for i in form_emestrada.exam.head():
    i = i.replace(".", "")
    i = re.sub(r"\s+", " ", i)
    parts = i.split(" ")
    i = " - ".join(parts[:-2]) + " - " + " ".join(parts[-2:])
    i = re.sub(r"reserva - (\d) ", r"reserva \1 ", i)
    print(i)

química - 2016 - junio - opción a
química - 2016 - junio - opción b
química - 2016 - reserva 1 - opción a
química - 2016 - reserva 1 - opción b
química - 2016 - reserva 2 - opción a


In [53]:
def format_exam(exam):
    exam = exam.replace(".", "")
    exam = re.sub(r"\s+", " ", exam)
    parts = exam.split(" ")
    exam = " - ".join(parts[:-2]) + " - " + " ".join(parts[-2:])
    exam = re.sub(r"reserva - (\d) ", r"reserva \1 ", exam)

    return exam

In [54]:
form_emestrada["exam"] = form_emestrada.exam.apply(format_exam)

In [55]:
form_emestrada.head()

,year,exam,statement
0,2016,química - 2016 - junio - opción a,a) Hidróxido de niquel(III); b) Ácido peryódic...
1,2016,química - 2016 - junio - opción b,a) Nitruro de aluminio; b) Hidrogenocromato de...
2,2016,química - 2016 - reserva 1 - opción a,Formule o nombre los siguientes compuestos: a)...
3,2016,química - 2016 - reserva 1 - opción b,Formule o nombre los siguientes compuestos: a)...
4,2016,química - 2016 - reserva 2 - opción a,Formule o nombre los siguientes compuestos: a)...


Limpiar texto de `statement`

In [65]:
def limpiar_formulacion(texto):
    if not pd.isna(texto):
        texto = texto.strip().replace(";", "").replace("\n"," ").replace(".", "")
        texto = texto.replace("Formule o nombre los siguientes compuestos: ", "").strip()
        return texto
    else:
        return texto

In [66]:
for st in form_emestrada.dropna().sample(3).statement.apply(limpiar_formulacion):
    print(f"{st}\n\n")

a) Fluoruro de cadmio b) Ácido selenioso c) Etanamida


a) Hidróxido de paladio(II) b) Ácido sulfúrico c) Ácido 2-aminopropanoico


a) Hidrogenocarbonato de sodio b) Peróxido de estroncio c) Nitrobenceno




In [71]:
for idx, row in form_emestrada.loc[~form_emestrada.statement.isna()].iterrows():
    print(f"{idx} - {limpiar_formulacion(row["statement"])}")

0 - a) Hidróxido de niquel(III) b) Ácido peryódico c) Nitrobenceno
1 - a) Nitruro de aluminio b) Hidrogenocromato de cobre(II) c) 3-metilbut-1-ino
2 - a) Óxido de vanadio(V) b) Ácido nitroso c) Metilpropeno
3 - a) Hidróxido de berilio b) Permanganato de bario c) Propanoato de metilo
4 - a) Carbonato de rubidio b) Sulfuro de cobre(II) c) Penta-1,3-dieno
5 - a) Óxido de plomo(IV) b) Ácido sulfuroso c) Etanoato de propilo
6 - a) Fluoruro de cadmio b) Ácido selenioso c) Etanamida
7 - a) Disulfuro de carbono b) Hidróxido de oro(III) c) 3-Clorofenol
8 - a) Óxido de platino(II) b) Sulfito de cadmio c) Ciclopenteno
9 - a) Hidruro de estaño(IV) b) Ácido carbónico c) Ácido 3-cloropropanoico
10 - a) Nitrato de hierro(II) b) Sulfuro de hidrógeno c) 1,3-Butenol
11 - a) Ácido nitroso b) Yoduro de níquel(II) c) 1,2-Etanodiol
12 - a) Ácido yódico b) Hidróxido de cromo(III) c) Butanal
13 - a) Óxido de mercurio(II) b) Clorato de potasio c) Ácido 3-metilhexanoico
14 - a) Hidrogenocarbonato de sodio b) Óx

In [72]:
form_emestrada.iloc[70]

year         2025
exam           - 
statement        
Name: 70, dtype: object

In [73]:
indices = form_emestrada[form_emestrada.statement == ""].index

In [58]:
df_emestrada.loc[:, "statement"] = df_emestrada.loc[:, "statement"].apply(limpiar_formulacion)

In [59]:
df_emestrada.sample(3)

,year,exam,statement
193,2019,"Química - 2019, Reserva, 1, Opción B",a) Hidruro de berilio b) Hidróxido de hierro(I...
21,2015,"Química - 2015, Reserva, 4, Opción B",a) Cloruro de calcio b) Óxido de níquel(II) c)...
24,2016,"Química - 2016, Junio, Opción A",a) Hidróxido de niquel(III) b) Ácido peryódico...


In [62]:
df_emestrada.to_csv("form_emestrada.csv", index = False, encoding="utf-8")

In [63]:
df_emestrada.year.aggregate(["min", "max"])

min    2000
max    2025
Name: year, dtype: int64